In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import os, json
import numpy as np
import scipy.sparse as sp
from pyscf import gto, scf
import sys; sys.path[:0] = ['..', '../data']  # repo root (srs/) and data/ (mol_info)
from srs.SRS_ray import SymmetryBasis, HamiltonianLoader, build_hamiltonian
from srs.encoding import stream_encode          # streaming Pauli encoder (srs/encoding.py)


def interleaved_to_blocked(x, M):
    """Mode map tau(2p) = p (alpha), tau(2p+1) = M + p (beta), applied bitwise."""
    x = np.asarray(x, dtype=np.int64)
    y = np.zeros_like(x)
    for p in range(M):
        y |= ((x >> (2 * p)) & 1) << p
        y |= ((x >> (2 * p + 1)) & 1) << (M + p)
    return y


def get_ham(Molecule, basis_sets='sto-3g', conv_tol=1e-12):
    """RHF reference (ROHF when spin != 0; Construction A) and integrals,
    relabelled to the blocked order alpha_0..alpha_{M-1} beta_0..beta_{M-1}."""
    mol = gto.M(atom=Molecule['atom'], basis=basis_sets,
                spin=Molecule['spin'], charge=Molecule['charge'], verbose=0)
    mf = scf.RHF(mol).run(conv_tol=conv_tol)
    if not mf.converged:
        raise RuntimeError("SCF not converged")
    M = mol.nao
    h1, h2, e_nuc = HamiltonianLoader(n_qubits=M).from_pyscf(mol, mf)
    tau = lambda k: k // 2 + M * (k % 2)
    h1B = {tuple(map(tau, k)): v for k, v in h1.items()}
    h2B = {tuple(map(tau, k)): v for k, v in h2.items()}
    return h1B, h2B, mol, mf, e_nuc


def build_block(h1B, h2B, mol, omega, mode='MHS'):
    """Restricted Hamiltonian (electronic) in the old Basis/Ham_setup order:
    ascending blocked integer label (colex)."""
    M, N = mol.nao, mol.nelectron
    states = np.asarray(SymmetryBasis(2 * M, N).build(mode=mode, multiplicity=omega))
    if states.size == 0:
        raise ValueError(f"{mode}(omega={omega}) is empty for N={N}, M={M}")
    basisB = np.sort(interleaved_to_blocked(states, M))
    Hb = build_hamiltonian(h1B, h2B, basisB, show=False)
    H = sp.csr_matrix((np.asarray(Hb.data), np.asarray(Hb.indices),
                       np.asarray(Hb.indptr)), shape=Hb.shape)
    if H.dtype != np.float64:
        raise ValueError(f"H has dtype {H.dtype}")
    return H, basisB


def auto_MHS_sections(Molecule, name='test', omegas=(0, 2), write=True,
                      shifts=('zero', 'mindiag'), outdir='encoded_hamiltonians',
                      export=True, n_workers=1, conv_tol=1e-12):
    """SRS_ray + stream_encode replacement of the qiskit-based version.
    export=False: numbers only (required for large D such as H22)."""
    h1B, h2B, mol, mf, e_nuc = get_ham(Molecule, conv_tol=conv_tol)
    records = []
    if write:
        os.makedirs(outdir, exist_ok=True)
    for omega in omegas:
        try:
            H, _ = build_block(h1B, h2B, mol, omega)
        except ValueError as e:
            print(f"  omega={omega}: skipped -- {e}")
            continue
        w = np.linalg.eigvalsh(H.toarray()) if H.shape[0] <= 4096 else None
        for shift in shifts:
            tag = '' if shift == 'zero' else '_shiftMinDiag'
            fname = (os.path.join(outdir, f"{name}_MHS_omega{omega}{tag}.txt")
                     if (write and export) else None)
            out = stream_encode(H, pad=shift, export=fname,
                                n_workers=1 if fname else n_workers, report=False)
            assert out['L_noI'] >= out['floor'] - 1e-8
            rec = dict(molecule=name, omega=omega, S=omega / 2, shift=shift,
                       constant=e_nuc, nuc=e_nuc, eigenval=w, **out)
            records.append(rec)
            print(f"  omega={omega} shift={shift:7s} lam={out['lam']:+.6f}: D={out['D']} "
                  f"q={out['q']} T={out['T']} c_I={out['c_I']:+.8f} L_noI={out['L_noI']:.6f} "
                  f"floor={out['floor']:.6f} E0={out['E0']:.8f} N_XBM={out['N_XBM']} Z={out['Z']:.6f}")
    if write:
        rpath = os.path.join(outdir, f"{name}_records.json")
        with open(rpath, 'w') as f:
            json.dump(records, f, indent=2,
                      default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
        print(f"saved {len(records)} records to '{rpath}'")
    return records

In [2]:
spacing = 1.0 
H22  = {"atom" : "; ".join(f"H 0 0 {i * spacing:.2f}" for i in range(22)),"charge" : 0, "spin" : 0}

In [ ]:
import time
from math import comb
import numpy as np, scipy.sparse as sp, ray

n, spacing = 22, 1.0                               # as in tab:H22
H22 = {"atom": "; ".join(f"H 0 0 {i * spacing:.2f}" for i in range(n)),
       "charge": 0, "spin": 0}

h1B, h2B, mol, mf, e_nuc = get_ham(H22, conv_tol=1e-12)
t0 = time.perf_counter()
H, basisB = build_block(h1B, h2B, mol, omega=0)
print(f"build {time.perf_counter() - t0:.1f} s, D={H.shape[0]}, nnz={H.nnz}")

# sanity checks before saving
M, N = mol.nao, mol.nelectron
assert H.shape[0] == comb(M, N // 2)
assert abs(H - H.T).max() == 0
occ = (1 << (N // 2)) - 1                          # orbitals 0..10 doubly occupied
hf_key = occ | (occ << M)                          # blocked label: alpha bits | beta bits << M
i = int(np.searchsorted(basisB, hf_key)); assert basisB[i] == hf_key
print(f"<HF|H|HF> + E_nuc - E_RHF = {H[i, i] + e_nuc - mf.e_tot:.2e}")

sp.save_npz("H22_MHS0.npz", H)
np.save("H22_MHS0_meta.npy", np.array([e_nuc, mf.e_tot]))
ray.shutdown()
print("saved")

[Loader] Loading from PySCF (RHF)
use tree


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[v] = "1"                      # before importing numpy

import json, time
import numpy as np, scipy.sparse as sp
import sys; sys.path[:0] = ['..']  # repo root (srs/)
from srs.encoding import stream_encode

H = sp.load_npz("H22_MHS0.npz")
e_nuc, e_rhf = np.load("H22_MHS0_meta.npy")
t0 = time.perf_counter()
res = stream_encode(H, pad='mindiag', n_workers=30, report= False)
res.update(wall_s=time.perf_counter() - t0, nuc=float(e_nuc), E_RHF=float(e_rhf),
           E_MHS_total=res['E0'] + float(e_nuc))
json.dump(res, open("H22_MHS0_stream.json", "w"), indent=2, default=float)
print(res)